# 12 · Training: attackers (E1, attacker-seed control, benchmark) and the ε pilot

Attackers are small policies that perturb the continuous latent of a neural codec (EnCodec 24 kHz or DAC 16 kHz)
within a fixed budget ε to raise a predictor's score, trained with GRPO for 2000 steps against a frozen predictor.
Each job below trains its attackers and then evaluates them, exactly like the original job
(about 30 min per job). The evaluation-only counterparts, which reuse the released attackers, are notebooks 21 and 23.

* `e1_<predictor>_s0..2` — fresh EnCodec and DAC attackers against every predictor (plus waveform PGD, the round-1
  attacker replayed, and random directions, which need no training).
* `e1seed_{R0,R3,R3ms}_s0` — five attackers with different seeds against one predictor.
* `bench_{sslmos,utmos,nisqa,dnsmos}` — five attackers against each published predictor.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first key); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1', 'e1_attacker_seeds', 'bench']]
print(len(KEYS), "runs:", ", ".join(KEYS))

31 runs: e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e1_R3ms_s0, e1_R3ms_s1, e1_R3ms_s2, e1seed_R0_s0, e1seed_R3_s0, e1seed_R3ms_s0, bench_sslmos, bench_utmos, bench_nisqa, bench_dnsmos


In [4]:
# Resolved configuration of the first run (checkpoint paths already point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: e1_attack
task: attack_eval
phase: main
seed: 0
run_name: e1_attack_R0_s0__2026-09-15_16-19-29
target_name: R0
target_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
transfer_attacker_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/hnp_s0.round1.attacker.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
e1:
  eval_set: bvcc/test
  max_utts: null
  judges:
  - utmos
  - dnsmos
attackers:
- name: grpo_encodec
  type: grpo
- name: grpo_dac
  type: grpo
  attack_overrides:
    codec:
      name: dac_16khz
      quantize: true
- name: pgd_wave
  type: pgd
  pgd:
    eps: 0.002
    steps: 50
    step_size: 0.0002
    batch_size: 8
- name: random_encodec
  type: random
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for e1_R0_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_e1_R0_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key e1_R0_s0



## Retrained vs released benchmark summaries

In [6]:
# Retrained vs released: the same summary file from both runs (where the retrained run has finished).
import json
rows = {}
released = repro.fetch_results()
for k in KEYS:
    new = repro.RUNS / "retrain" / k / "results/bench_summary.json"
    old = released / Path(reg[k]["run_dir"]).name / "results/bench_summary.json"
    if new.exists() and old.exists():
        rows[(k, "released")] = pd.json_normalize(json.loads(old.read_text())).iloc[0]
        rows[(k, "retrained")] = pd.json_normalize(json.loads(new.read_text())).iloc[0]
pd.DataFrame(rows).T['target', 'asr_ref_recon_mean', 'asr_ref_recon_max', 'gain_ref_recon_mean', 'recon_gain_mean'] if rows else "no finished retrained runs yet"

'no finished retrained runs yet'

## ε pilot (validation split only)

Chooses the perturbation budget before any test data is seen: attackers with ε ∈ {0.5, 1, 2} against R0 seed 0,
2000 steps, 400 BVCC validation utterances. The paper quotes the resulting ceiling rates
(`pilot_ceiling_eps*` in `analysis/out/paper_numbers.json`). The job writes `pilot_eps3/h0_e{ε}.json`.

In [7]:
import subprocess
r0 = repro.remap(repro.registry()["r0_s0"]["run_dir"] + "/results/selected_checkpoint.txt")
repro.fetch(["r0_s0"])
out_dir = repro.RUNS / "retrain" / "pilot_eps3"
out_dir.mkdir(parents=True, exist_ok=True)
cmds = [f"python scripts/probe_attack.py --ckpt {r0} --config repro/configs/pilot_eps3_hack_loop.yaml "
        f"--eps {eps} --steps 2000 --n_val 400 --out {out_dir}/h0_e{eps}.json "
        f"--overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere" for eps in ("0.5", "1.0", "2.0")]
for c in cmds:
    script = (f"#!/bin/bash\n#SBATCH --account=PAS2966\n#SBATCH --partition=nextgen\n#SBATCH --time=01:00:00\n"
              f"#SBATCH --gpus-per-node=1\n#SBATCH --cpus-per-task=12\n#SBATCH --mem=64G\n"
              f"#SBATCH --output={out_dir}/%j.out\nset -euo pipefail\ncd {ROOT}\nsource scripts/env.sh\n{c}\n")
    if LAUNCH:
        print(subprocess.run(["sbatch", "--parsable"], input=script, capture_output=True, text=True).stdout.strip())
    else:
        print(c)

[fetch] 1 checkpoints ready in /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints


python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_hack_loop.yaml --eps 0.5 --steps 2000 --n_val 400 --out /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/pilot_eps3/h0_e0.5.json --overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere
python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_hack_loop.yaml --eps 1.0 --steps 2000 --n_val 400 --out /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/pilot_eps3/h0_e1.0.json --overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere
python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_ha

In [8]:
import json
rel = repro.fetch_results() / "pilot_eps3"
rows = []
for eps in ("0.5", "1.0", "2.0"):
    for src, d in (("released", rel), ("retrained", out_dir)):
        f = d / f"h0_e{eps}.json"
        if f.exists():
            r = json.loads(f.read_text())[0]
            rows.append({"eps": eps, "source": src, "val_ceiling_rate@0.3": r["val_ceiling_rate@0.3"],
                         "val_mean_gain": r["val_mean_gain"]})
pd.DataFrame(rows)

,eps,source,val_ceiling_rate@0.3,val_mean_gain
0,0.5,released,0.2225,0.090977
1,1.0,released,0.2150,0.071680
2,2.0,released,0.1850,0.052383
